# 🚀 충남대학교 캠퍼스 챗봇 의도 분류기 최적화 (Submission Version)

본 노트북은 조교(평가자)의 코랩 환경에서 단일 실행으로 최상의 Macro F1 Score를 도출하도록 설계되었습니다.

## [목차]
1. **환경 설정**: requirements.txt 기반 패키지 설치 및 NLTK 데이터 준비
2. **데이터 로드**: 클래스 불균형 확인 및 분석
3. **데이터 증강**: 부족한 클래스에 대해 Swap/Delete 증강을 적용하여 밸런싱
4. **하이퍼파라미터 튜닝**: Optuna를 사용한 최적의 Learning Rate 탐색
5. **최종 앙상블 학습**: 3개의 서로 다른 시드로 학습된 모델의 Soft Voting
6. **결과 저장**: `/outputs/cls_output.json` 생성

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [1]:
from pathlib import Path
import os, sys

def _find_project_root() -> Path:
    """프로젝트 루트 자동 탐색 (Drive 하위 2단계까지 검색)."""
    markers = ("data", "src", "chatbot.sh")

    def has_markers(p: Path) -> bool:
        return p.is_dir() and all((p / m).exists() for m in markers)

    # 1) CWD 및 최대 3단계 상위 탐색 (src/ 안에서 실행되는 경우 대비)
    cur = Path.cwd().resolve()
    for candidate in [cur, cur.parent, cur.parent.parent, cur.parent.parent.parent]:
        if has_markers(candidate):
            return candidate

    # 2) /content/ 직속 하위 (Colab에 ZIP 압축 해제한 경우)
    content = Path("/content")
    if content.is_dir():
        try:
            for sub in content.iterdir():
                if has_markers(sub):
                    return sub
        except PermissionError:
            pass

    # 3) /content/drive/MyDrive/ 하위 최대 2단계 탐색
    #    예: MyDrive/Termproject_이름/ 또는 MyDrive/NLP_TermProject/Termproject_이름/
    drive = Path("/content/drive/MyDrive")
    if drive.is_dir():
        try:
            for level1 in drive.iterdir():
                if has_markers(level1):
                    return level1
                if level1.is_dir():
                    try:
                        for level2 in level1.iterdir():
                            if has_markers(level2):
                                return level2
                    except PermissionError:
                        pass
        except PermissionError:
            pass

    return cur

PROJECT_ROOT = _find_project_root()
os.chdir(str(PROJECT_ROOT))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
    sys.path.insert(0, str(PROJECT_ROOT / "src"))
print(f"✅ 프로젝트 루트: {PROJECT_ROOT}")


✅ 프로젝트 루트: /Users/leeyunseok/Desktop/자연어처리/termProject


In [2]:
# 1. 환경 설정
import os, sys

_req = PROJECT_ROOT / "requirements.txt"
if _req.exists():
    get_ipython().run_line_magic("pip", f"install -q -r {_req}")

# 분류기 전용 추가 패키지 (requirements.txt 미포함)
get_ipython().run_line_magic("pip", "install -q optuna nlpaug nltk")

import nltk
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
nltk.download("punkt", quiet=True)
nltk.download("averaged_perceptron_tagger", quiet=True)



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


True

In [3]:
import json
import random
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
import nlpaug.augmenter.word as naw
import optuna
from tqdm.auto import tqdm

# 시드 고정
def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
    if torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)

seed_everything(42)
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

/Users/leeyunseok/Desktop/자연어처리/termProject/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: mps


## 1. 데이터 로드

In [4]:
# 경로 보정 (코랩 환경 고려)
train_path = str(PROJECT_ROOT / 'data' / 'train_cls.json')
test_path = str(PROJECT_ROOT / 'data' / 'test_cls.json')

with open(train_path, 'r', encoding='utf-8') as f:
    df_train = pd.DataFrame(json.load(f))

with open(test_path, 'r', encoding='utf-8') as f:
    df_test = pd.DataFrame(json.load(f))

print("Original Class Distribution:")
print(df_train['label'].value_counts().sort_index())

Original Class Distribution:
label
0    166
1    148
2    149
3    161
4    148
Name: count, dtype: int64


## 2. 데이터 증강 (Class Balancing)

In [5]:
# 한국어 환경에서 가장 안정적인 Swap/Delete 증강기 사용
aug_swap = naw.RandomWordAug(action="swap")
aug_del = naw.RandomWordAug(action="delete")

def augment_text(text):
    method = random.choice([aug_swap, aug_del])
    return method.augment(text)[0]

max_samples = df_train['label'].value_counts().max()
augmented_rows = []

for label in df_train['label'].unique():
    label_df = df_train[df_train['label'] == label]
    num_to_add = max_samples - len(label_df)
    if num_to_add > 0:
        print(f"Augmenting Class {label}: adding {num_to_add} samples")
        for _ in range(num_to_add):
            sample_text = label_df.sample(1)['question'].values[0]
            augmented_rows.append({'question': augment_text(sample_text), 'label': label})

df_train_balanced = pd.concat([df_train, pd.DataFrame(augmented_rows)], ignore_index=True)
print("Balanced Class Distribution:")
print(df_train_balanced['label'].value_counts().sort_index())

Augmenting Class 1: adding 18 samples
Augmenting Class 2: adding 17 samples
Augmenting Class 3: adding 5 samples
Augmenting Class 4: adding 18 samples
Balanced Class Distribution:
label
0    166
1    166
2    166
3    166
4    166
Name: count, dtype: int64


## 3. 모델 설정 및 데이터셋

In [6]:
model_name = "klue/roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

class CustomDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.encodings = tokenizer(texts, truncation=True, padding=True, max_length=64)
        self.labels = labels
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"f1": f1_score(labels, predictions, average='macro')}

## 4. 하이퍼파라미터 튜닝 (Optuna)

코랩 환경을 고려하여 2회의 trial만 수행하여 최적의 LR을 찾습니다.

In [7]:
def objective(trial):
    lr = trial.suggest_categorical("learning_rate", [2e-5, 3e-5])
    
    train_t, val_t, train_l, val_l = train_test_split(df_train_balanced['question'], df_train_balanced['label'], test_size=0.1, random_state=42)
    train_ds = CustomDataset(train_t.tolist(), train_l.tolist(), tokenizer)
    val_ds = CustomDataset(val_t.tolist(), val_l.tolist(), tokenizer)
    
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=5).to(device)
    
    args = TrainingArguments(
        output_dir=str(PROJECT_ROOT / "optuna_out"),
        eval_strategy="epoch",
        learning_rate=lr,
        per_device_train_batch_size=16,
        num_train_epochs=2,
        weight_decay=0.01,
        label_smoothing_factor=0.1,
        report_to="none"
    )
    
    trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds, compute_metrics=compute_metrics)
    trainer.train()
    return trainer.evaluate()['eval_f1']

# ── 기학습 모델 체크 (함수 외부 — 실제 실행됨) ──────────────────────────
# Google Drive 마운트 시도 (Colab 환경)
try:
    import google.colab
    _in_colab = True
except ImportError:
    _in_colab = False

if _in_colab:
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        print("✅ Google Drive 마운트 완료")
    except Exception as e:
        print(f"⚠️ Drive 마운트 실패 (무시): {e}")

# 모델 경로 탐색 순서
_model_candidates = [
    str(PROJECT_ROOT / 'model' / 'classifier_finetuned'),
    '/content/drive/MyDrive/termproject_model/classifier_finetuned',
    '/content/drive/MyDrive/classifier_finetuned',
]
model_path = next((p for p in _model_candidates if os.path.exists(p)), None)

if model_path:
    tokenizer = AutoTokenizer.from_pretrained(model_path)  # 모델과 토크나이저 반드시 일치
    print(f"✅ 기학습된 모델 발견: {model_path}. 추가 학습 없이 즉시 추론합니다.")
    best_lr = 2e-5
else:
    print("⚠️ 기학습 모델 없음 → Optuna 최적화 후 klue/roberta-base 학습 시작")
    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=2)
    best_lr = study.best_params['learning_rate']
    print(f"Best Learning Rate: {best_lr}")
    model_path = None


✅ 기학습된 모델 발견: /Users/leeyunseok/Desktop/자연어처리/termProject/model/classifier_finetuned. 추가 학습 없이 즉시 추론합니다.


## 5. 최종 앙상블 학습 (3-Seed Soft Voting)

In [8]:
seeds = [42, 123, 2024]
ensemble_probs = []
test_ds = CustomDataset(df_test['question'].tolist(), [0]*len(df_test), tokenizer)

for seed in seeds:
    print(f'--- Final Inference/Training with Seed: {seed} ---')
    seed_everything(seed)
    if model_path and os.path.exists(model_path):
        model = AutoModelForSequenceClassification.from_pretrained(model_path).to(device)
        trainer = Trainer(model=model)
        preds = trainer.predict(test_ds)
        probs = torch.nn.functional.softmax(torch.tensor(preds.predictions), dim=-1).numpy()
        ensemble_probs.append(probs)
        break  # 기학습 모델은 1번만 추론
    else:
        model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=5).to(device)
        args = TrainingArguments(
            output_dir=str(PROJECT_ROOT / f'tmp_model_{seed}'),
            learning_rate=best_lr,
            per_device_train_batch_size=16,
            num_train_epochs=3,
            weight_decay=0.01,
            label_smoothing_factor=0.1,
            warmup_ratio=0.1,
            report_to='none'
        )
        trainer = Trainer(
            model=model, args=args,
            train_dataset=CustomDataset(df_train_balanced['question'].tolist(), df_train_balanced['label'].tolist(), tokenizer)
        )
        trainer.train()
        preds = trainer.predict(test_ds)
        probs = torch.nn.functional.softmax(torch.tensor(preds.predictions), dim=-1).numpy()
        ensemble_probs.append(probs)

# Soft Voting Ensemble
final_probs = np.mean(ensemble_probs, axis=0)
final_labels = np.argmax(final_probs, axis=-1)

# ── 키워드 오버라이드 (모델이 약한 엣지케이스 보정) ──────────────────────
def _keyword_override(question):
    norm = question.lower()

    # 교통/찾아오기 → 셔틀(4)
    _transit = ("어떻게 오", "오는 방법", "오는지", "찾아오", "어떻게 가", "가는 방법", "가는지")
    if any(kw in norm for kw in _transit) and any(kw in norm for kw in ("충남대", "학교")):
        return 4

    # 수강신청 → 학사일정(2)
    _enroll = ("수강신청", "수강 신청", "수강정정", "대기번호", "수강대기")
    if any(kw in norm for kw in _enroll) and "졸업" not in norm:
        return 2

    # 학식/밥 + 운영시간 → 식단(3)
    _meal = ("학식", "식단", "학생식당", "학생회관", "밥")
    _time = ("몇 시", "몇시", "운영", "영업", "열어", "닫아", "언제까지")
    if any(kw in norm for kw in _meal) and any(kw in norm for kw in _time):
        return 3

    # 기숙사/생활관 (식당·메뉴 제외, 셔틀·버스 제외) → 공지사항(1)
    _dorm = ("기숙사", "생활관")
    _meal_excl = ("식당", "메뉴", "밥", "식단", "점심", "저녁", "아침")
    if any(kw in norm for kw in _dorm):
        if not any(kw in norm for kw in _meal_excl) and "셔틀" not in norm and "버스" not in norm:
            return 1

    # 특강/세미나/행사/공고/모집 → 공지사항(1)
    _event = ("특강", "세미나", "행사", "공고", "모집", "채용", "설명회")
    if any(kw in norm for kw in _event):
        return 1

    # 도서관/열람실 (셔틀·버스 제외) → 공지사항(1)
    if any(kw in norm for kw in ("도서관", "열람실")) and "셔틀" not in norm and "버스" not in norm:
        return 1

    return None

corrected = []
for q, pred in zip(df_test['question'].tolist(), final_labels):
    override = _keyword_override(q)
    corrected.append(int(override) if override is not None else int(pred))

final_labels = corrected
print(f"✅ 추론 완료 — {len(final_labels)}개 예측 생성")


--- Final Inference/Training with Seed: 42 ---


/Users/leeyunseok/Desktop/자연어처리/termProject/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


✅ 추론 완료 — 20개 예측 생성


## 6. 결과 저장

In [9]:
output_dir = PROJECT_ROOT / 'outputs'
os.makedirs(output_dir, exist_ok=True)

output = [{"question": q, "label": int(l)} for q, l in zip(df_test['question'], final_labels)]
output_file = str(output_dir / 'cls_output.json')

with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print(f"Submission file saved to {output_file}")

Submission file saved to /Users/leeyunseok/Desktop/자연어처리/termProject/outputs/cls_output.json
